# Panel Data Analysis in Python
- Simulate a panel data and causal model 
- OLS (taking panel data as cross-sectional)
- Within estimator 
   - check if any observations are dropped for within estimation
- Fixed effects: 
   - verify if we drop individuals with constant $x_{it}\equiv x_i$, the estimate is the same
   - check if $\alpha_i$ for those with $x_{it}\equiv x_i$ just equal to $\bar{y}$

In [ ]:
import pandas as pd # for data manipulation
import numpy as np  # to work with arrays (vectors/matrices)
import os  # for setting directory 
print(os.getcwd())
# os.chdir() # input your personal directory where the dataset is saved
import statsmodels.formula.api as smf # for OLS regressions
from linearmodels import PanelOLS

In [ ]:
## 1. Set up a panel data and specify the data generating process
N = 1000; # num. units (individuals)
T = 10 ; # num. periods 

tmp_i = pd.DataFrame(range(N),columns=['i']); tmp_i['1']=1 
tmp_t = pd.DataFrame(range(T),columns=['t']); tmp_t['1']=1 

panel = pd.merge(tmp_i, tmp_t, on=['1'], how='inner')
panel = panel.drop(columns=['1']) 

assert panel.shape[0] == N*T

## 0. Data Generating Process
- Covariate of interest: $x_{it}~ Bernoulli(0.5)$. 
   - We reassigned 50 units to have $x_{it}\equiv 0$, and another 50 units to have $x_{it}\equiv 1$. 
- Time-invariant unobserved "ability": $\alpha_i ~ Exponential(1)$
- Model: $y_{it} = \beta \times x_{it} + \alpha_i + \epsilon_{it}$, where $\epsilon_{it} \sim N(0,1)$.  

In [ ]:
np.random.seed(50) # set the seed so you get the same (set of) random numbers when re-running the code below

# Covariate x_it: 0/1  ~ bernoulli(0.5)  
panel['x'] = np.random.binomial(n=1, p = 0.5, size=panel.shape[0]) # x 

# assign some units to have x_it always 0 or always 1: 
panel.loc[panel['i']<=50,'x']=0 
panel.loc[panel['i'].between(51,100),'x']=1 

# compute xbar (x_i) = sample mean of "x" for each unit" 
tmp = panel.groupby(['i'])['x'].mean().reset_index(drop=False); tmp.columns=['i','xbar']
panel = pd.merge(panel, tmp, on=['i'], how='left')
assert 0== panel[['i','t']].duplicated().sum() 

l_always0 = panel.loc[panel['xbar']==0,'i'].unique().tolist()
l_always1 = panel.loc[panel['xbar']==1,'i'].unique().tolist() 
print(f'num. with x always 0: {len(l_always0)}, num. with x always 1: {len(l_always1)}') 


Ability $\alpha_i$ is correlated with $\bar{x}_i$.
 - Conditional on $\bar{x}_i\geq 0.5$, $\alpha_i \sim Exponential(1)$ with mean 1. 
 - Conditional on $\bar{x}_i<\geq> 0.5$, $\alpha_i \sim Exponential(0.2)$ with mean 5.

Recall exponential distribution: $\alpha~Exponential(\lambda)$ has density function $f(\alpha;\lambda) = \lambda\times e^{-\lambda *\alpha}$ for $\alpha \geq 0 $. It has mean $\frac{1}{\lambda}$ and variance $\frac{1}{\lambda^2}$. 

In np.random.exponential, scale = $\frac{1}{\lambda}$. https://numpy.org/doc/2.1/reference/random/generated/numpy.random.exponential.html

In [ ]:
# Ability at "i" level: alpha ~ standard normal 
# make alpha correlated with xbar ("x_i"): 
tmp_i['alpha'] =  np.random.exponential(scale= 1, size= N)  # unique at "i" level 
tmp_i['alpha_'] = np.random.exponential(scale= 5, size= N) 

panel = pd.merge(panel[['i','t','x','xbar']], tmp_i[['i','alpha','alpha_']], how='left')
assert 0 == panel[['i','t']].duplicated().sum()

panel.loc[panel['xbar']>=0.5,'alpha'] = panel.loc[panel['xbar']>=0.5,'alpha_'].copy() 
print(panel[['x','xbar','alpha']].corr())




In [ ]:
# model 
beta = 0.5 
panel['y'] = beta * panel['x'] + panel['alpha'] + np.random.normal(loc=0, scale=2, size= N*T) 

## 1. OLS
$$ y_{it} = \hat{b}^{OLS}_0 + \hat{\beta}^{OLS}_1 + \hat{u}_{it}$$ 

In [ ]:
model = smf.ols(f"y ~ 1+ x", data=panel).fit(cov_type='HC1')
print(model.summary())

# compare coef on x vs. beta:
print(model.params['x'].item(), beta)

## 2. Within 
$$\hat{\beta}^{W} = \frac{\sum_{i,t} (x_{it}-\bar{x}_i)(y_{it}-\bar{y}_i)}{\sum_{i,t} (x_{it}-\bar{x}_i)^2} $$
Two questions: 
- when we estimate $\hat{\beta}^{W}$, does smf.ols drop one of period for each unit, given that  $\sum_{t} (x_{it}-\bar{x}_i) = 0$ wihtin each unit $i$? No. 

In [ ]:
# Within transformation: note we already have xbar, need ybar:
tmp = panel.groupby(['i'])['y'].mean().reset_index(drop=False); tmp.columns=['i','ybar']
panel = pd.merge(panel, tmp, on=['i'], how='left')
assert 0==panel[['i','t']].duplicated().sum() 

# de-meaned within each unit: 
panel['d_y'] = panel['y'] - panel['ybar']
panel['d_x'] = panel['x'] - panel['xbar']

# verify sum(d_y)=sum(d_x)= 0 within each unit:
print(panel.groupby(['i'])[['d_y','d_x']].sum().describe())


In [ ]:
# Now let's get the within estimator, pay attention to the size of the estimation sample/warnings:
model_w = smf.ols(f"d_y ~ 1+ d_x", data=panel).fit(cov_type='HC1')
print(model_w.summary())


No observation is dropped in the estimation for $\hat{\beta}^{W}$! 
Correction from lecture on 10/29/2025: it only makes sense to drop 1 period if T=2 -- we will look at this special case more closely on 11/03/2025! 

Let's double check $\sum_{i,t} (x_{it}-\bar{x}_i)^2$ is not zero (invertible if you have more than 1 covariate): 

In [ ]:
print((panel['d_x']**2).mean())

In [ ]:
# If we drop a period ourselves, "smf" doesn't understand we have done a within transformation, and the answer would be different. 
model_w2= smf.ols(f"d_y ~ 1+ d_x", data=panel.loc[panel['t']!=0]).fit(cov_type='HC1')
print(model_w2.params)
print(model_w.params) # correct 

What about individuals who have $x_{it}\equiv 0$ and $x_{it} \equiv 1$? They have $d_x = x_{it} - \bar{x}_i \equiv 0$. 

$\hat{\beta}_1^{W}$ is identified from *within*-person variation in $x_{it}$. 

In [ ]:
# double check d_x always 0 if units are in l_always0+l_always1 
print(panel.loc[panel['i'].isin(l_always0+l_always1), 'd_x'].describe())

model_w3 = smf.ols(f"d_y ~ 1+ d_x", data=panel.loc[~panel['i'].isin(l_always0+l_always1)]).fit(cov_type='HC1')
print(model_w3.params)
print(model_w.params) # correct 

## 3. Fixed Effects
Recall the model is: 
$$ y_{it} = \beta_1 x_{it} + \alpha_i + \epsilon_{it}$$ 
We estimate a regression of $y_{it}$ on $x_{it}$ with person ($i$) fixed effects. 

- You may or may not include a constant. The coefficient of interest is $\beta_1$ on $x_{it}$. We will check if we get the same $\hat{beta}_1$ with or without a constant. 
- What happens to individuals with $x_{it}\equiv 1$ or $x_{it}\equiv 0$? 

In [ ]:
# This automatically absorbs fixed effects without showing them
model_FE= PanelOLS.from_formula("y ~ 1 +x + EntityEffects", 
                               data=panel.set_index(['i','t'])).fit(cov_type='clustered', cluster_entity=True) 
print(model_FE)
print(model_FE.params)
# coef_FE = model_FE.params[['educ','exp2','married']]

In [ ]:
# suppose we drop a constant, the coeffienct on x should be the same 
model_FE_ = PanelOLS.from_formula("y ~ 0 +x + EntityEffects", 
                               data=panel.set_index(['i','t'])).fit(cov_type='clustered', cluster_entity=True) 
print(model_FE_)
print(model_FE_.params)

In [ ]:
# compare mean fixed effects above with the coefficient on constant in model_FE: 
fe = model_FE.estimated_effects
print(fe.shape, fe.mean())

fe_ = model_FE_.estimated_effects
print(fe_.shape, fe_.mean())

# recall intercept in the model with 1 (model_FE):
print(model_FE.params['Intercept'])

Now let's see what were the estimated fixed effects on those with $x_{it}\equiv 0$ or 1. 

In the model w/o constant, do we have $\hat{\alpha}_i = \bar{y}_i$? 

In [ ]:
# Merge in fixed effects from the model w/o constant:

fe_ = fe_.reset_index(drop=False) 
panel = pd.merge(panel, fe_, on=['i','t'], how='left')
assert 0== panel[['i','t']].duplicated().sum() 


In [ ]:
# diff between estimated fixed effect and ybar: 
panel['check'] = panel['ybar'] - panel['estimated_effects']
print(panel.loc[panel['i'].isin(l_always0),'check'].describe())

print(panel.loc[panel['i'].isin(l_always1),'check'].describe()) # mean  = beta1! 

# print(panel.loc[~panel['i'].isin(l_always0+l_always1),'check'].describe())

In [ ]:
print(panel[['ybar','estimated_effects']].head())

In [ ]:
# what if we drop always0/1: 
model_FE2= PanelOLS.from_formula("y ~ x + EntityEffects", 
                               data=panel.loc[~panel['i'].isin(l_always0+l_always1)].set_index(['i','t'])).fit(cov_type='clustered', cluster_entity=True) 
print(model_FE2) # same estimate!

In [ ]:
# and what if we estimate the regression only on always0/1: 
model_FE3= PanelOLS.from_formula("y ~ x + EntityEffects", 
                               data=panel.loc[panel['i'].isin(l_always0+l_always1)].set_index(['i','t'])).fit(cov_type='clustered', cluster_entity=True) 
print(model_FE3) 